# Libraries

In [301]:
# Stat Libs
import pandas as pd
from pandas.api.types import CategoricalDtype
import pickle
import re

# Stat Libs
import statsmodels.api as sm
from itertools import product
from functools import reduce

# Statistical libs
from sklearn.preprocessing import MultiLabelBinarizer


# Load Data
- No reset index is applied into saved dfs --> Models have same indexes as intial dfis from thesis_data.ipynb

In [ ]:
df1 = pd.read_pickle(r".\df_data\df1.pkl")
df2 = pd.read_pickle(r".\df_data\df2.pkl")
df3 = pd.read_pickle(r".\df_data\df3.pkl")
df4 = pd.read_pickle(r".\df_data\df4.pkl")

df_with = pd.read_pickle(r".\df_data\df_with.pkl")

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\Eugenia\AppData\Local\Temp\ipykernel_19596\3596763811.py:1: SyntaxWarning: invalid escape sequence '\d'
  '''df1 = pd.read_pickle(r".\df_vizual\df1_vizual.pkl")


In [303]:
df1.columns.to_list()

['Study_Status_Bin',
 'Sex_List',
 'Age_List',
 'Enrollment_Counts',
 'Funder_Industry_Bin',
 'Funder_Counts',
 'Funder_Counts_Log',
 'Funder_Bin',
 'Start_Date_Year_Counts',
 'Completion_Date_Year_Counts',
 'Completion_Gap_Counts',
 'Completion_Gap_Log',
 'Intervention_Type_List',
 'Intervention_Type_Counts',
 'Intervention_Route_List',
 'Intervention_Route_Counts',
 'Placebo_Bin',
 'Standard_Care_Bin',
 'Healthy_Bin',
 'Covid_19_Bin',
 'Conditions_Detail_List',
 'Adverse_List',
 'Adverse_Counts',
 'Adverse_Counts_Log',
 'Adverse_Bin',
 'Adverse_System_List',
 'Adverse_System_Counts',
 'Adverse_System_Counts_Log',
 'Allocation_Bin',
 'Intervention_Model_List',
 'Masking_List',
 'Masking_Detail_List',
 'Primary_Purpose_List',
 'Arm_Counts',
 'Arm_Counts_Log',
 'Continents_List',
 'City_Counts',
 'Country_Counts',
 'Continent_Counts',
 'Enrollment_Counts_Log',
 'City_Counts_Log',
 'Continent_Counts_Log',
 'Country_Counts_Log',
 'Global_Bin',
 'Enrollment_Categ',
 'Adverse_System_Categ',

# Drop cols

In [304]:
def fun_drop(dfi):   
    dfi = dfi.drop(columns = ['Enrollment_Counts', 'Enrollment_Categ', 
                              
                                'City_Counts', 'City_Categ',
                                'Country_Counts', 'Country_Counts_Log',
                                'Continent_Counts',  'Continent_Counts_Log', 'Continent_Categ', # all dropped

                                'Funder_Counts', 'Funder_Bin',# Keep bin --> good median --> barplots overplap
                                'Adverse_Counts', 'Adverse_Bin', 
                                'Adverse_System_List', 'Adverse_System_Counts', 'Adverse_System_Counts_Log', 'Adverse_System_Categ',# all dropped

                                'Arm_Counts', 'Arm_Categ',

                                'Intervention_Route_Counts', 'Intervention_Route_Categ', # all dropped
                                'Intervention_Type_Counts', 'Intervention_Type_Categ', # all dropped

                                'Masking_Detail_List',
                                'Covid_19_Bin', # all dropped

                                'Start_Date_Year_Counts', 'Start_Date_Year_Categ',
                                'Completion_Date_Year_Counts', 'Completion_Date_Year_Categ',
                                'Completion_Gap_Counts', 'Completion_Gap_Log', 'Completion_Gap_Categ'], axis=1)
    return dfi

df1 = fun_drop(df1)
df2 = fun_drop(df2)
df3 = fun_drop(df3)
df4 = fun_drop(df4)

df_with = df_with.drop(columns = ['Enrollment_Counts',
                                  
                                  'City_Counts', 
                                  'Country_Counts', 'Country_Counts_Log',
                                  'Continent_Counts', 'Continent_Counts_Log',

                                  'Funder_Counts', 'Funder_Bin',
                                  'Adverse_Counts', 'Adverse_Bin', 
                                  'Adverse_System_List', 'Adverse_System_Counts', 'Adverse_System_Counts_Log', 

                                  'Arm_Counts', 
                                  
                                  'Intervention_Route_Counts', 
                                  'Intervention_Type_Counts', 

                                  'Masking_Detail_List', 
                                  'Covid_19_Bin', 

                                  'Completion_Gap_Counts', 'Completion_Gap_Log', 
                                  'Start_Date_Year_Counts', 'Completion_Date_Year_Counts', 
                                  
                                  'Phases_List'])


In [305]:
display(df1.shape)
display(df2.shape)
display(df3.shape)
display(df4.shape)
display(df_with.shape)

(21012, 22)

(19747, 22)

(13317, 22)

(12420, 22)

(60717, 22)

In [306]:
dfis = [df1, df2, df3, df4, df_with]
iss = [1, 2, 3, 4, 5]

# Dummies
"_List” data: List element data, were encoded through dummy creation. The problem was that list-element data cannot be used from models. Firstly, list-element rows were exploded to one row per element of list. However, this inflates sample size, as rows increase and duplicate for same nct_id if more than one element occur in a list-row. For this reason, these data were then grouped by nct_id, so number of rows remained the same as initial datasets (df0, df1, df2, df3, df4, df5).

In [307]:
# Alternatively for List element columns 
mlb = MultiLabelBinarizer()

In [308]:
## Dummies
def fun_dum_enc(dfi, cols):
    for col in cols:  
        df_expl = dfi.copy()
        df_expl = df_expl.explode(col)

        df_expl[col] = df_expl[col].astype('category') 
        df_expl[col] = df_expl[col].cat.remove_unused_categories()
        df_expl[col] = df_expl[col].astype('str') #str cause of error in encoding. After astype(cat) so to drop unused categories

        dummies = pd.get_dummies(df_expl[col], drop_first = False, dtype = int, prefix = col , prefix_sep='_')
        
        dummies.index = df_expl.index # ensure same indexing with df_expl
        dummies = dummies.groupby(dummies.index).sum().clip(upper = 1) 
        # clip: if a row has double entry data ['UNSPES', 'UNSPES'] it avoids double vounting with sum().

        dfi = pd.concat([dfi.drop(columns = [col], axis = 1), dummies], axis = 1)  
    return dfi

### Cols
def fun_dum_cols(dfis):  # In case they are not the same.
    dum_cols = []
    for dfi in dfis: # loop inputed in case dfis have not all the same columns. # * Plus not to run function into funtion.
        dum_cols = dum_cols + [[col for col in dfi.columns if '_List' in col]]
    return dum_cols

dum_cols = fun_dum_cols(dfis) 

# Apply
# * loop so not to run function into function
df1 = fun_dum_enc(df1, dum_cols[0])
df2 = fun_dum_enc(df2, dum_cols[1])
df3 = fun_dum_enc(df3, dum_cols[2])
df4 = fun_dum_enc(df4, dum_cols[3])
df_with = fun_dum_enc(df_with, dum_cols[4])

# Example
display(dum_cols[0])
df1[[col for col in df1.columns if '_list' in col.lower()]] #.head()  # Transposed for better view


['Sex_List',
 'Age_List',
 'Intervention_Type_List',
 'Intervention_Route_List',
 'Conditions_Detail_List',
 'Adverse_List',
 'Intervention_Model_List',
 'Masking_List',
 'Primary_Purpose_List',
 'Continents_List']

,Sex_List_FEMALE,Sex_List_MALE,Age_List_ADULT,Age_List_CHILD,Age_List_OLDER_ADULT,Intervention_Type_List_BIOLOGICAL,Intervention_Type_List_DEVICE,Intervention_Type_List_DRUG,Intervention_Type_List_INTERV_OTHER,Intervention_Type_List_INTERV_UNSPES,...,Masking_List_SINGLE,Masking_List_TRIPLE,Primary_Purpose_List_DIAGNOSTIC,Primary_Purpose_List_PREVENTION,Primary_Purpose_List_PRIM_PURP_OTHER,Primary_Purpose_List_TREATMENT,Continents_List_Asia,Continents_List_Cont_Other,Continents_List_Europe,Continents_List_North America
0,0,1,1,0,0,0,0,1,0,0,...,0,0,0,0,0,1,1,0,0,0
1,0,1,1,0,0,0,0,1,0,0,...,0,0,1,0,0,0,1,0,0,0
5,1,1,1,0,1,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,1
6,1,1,1,0,0,1,0,0,0,0,...,0,1,0,0,1,0,0,0,0,1
7,1,1,1,0,1,0,0,1,0,0,...,0,0,0,0,0,1,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66481,1,1,1,1,0,0,0,1,0,0,...,0,0,0,0,1,0,0,0,0,1
66489,1,1,1,0,1,0,0,0,0,1,...,0,0,1,0,0,0,0,0,0,1
66492,1,1,1,0,0,0,0,1,0,0,...,0,0,0,0,0,1,1,0,0,0
66495,1,1,1,0,0,0,0,1,0,0,...,0,1,0,0,0,1,0,0,0,1


## Drop First
- Drop first was not done through get_dummies command, so to choose the column to drop, based on its characterisic. e.g., drop phase 0 is preferred that to drop phase 3.


In [309]:
dfis = [df1, df2, df3, df4, df_with]
iss = [1, 2, 3, 4, 5]

In [310]:

def fun_drop_first(dfi):
    dum_cols = [col for col in dfi.columns if '_list' in col.lower()]
    
    keywords = ['other', 'none', 'Sex_Bin_all', 'age_list_child', 'na_randomize', 'single_grou', 'female']
    drop_cols = [col for col in dum_cols if any(key in col.lower() for key in keywords) and '_x_' not in col]
    
    dfi = dfi.drop(columns = drop_cols, axis = 1)
    return dfi

df1 = fun_drop_first(df1)
df2 = fun_drop_first(df2)
df3 = fun_drop_first(df3)
df4 = fun_drop_first(df4)
df_with = fun_drop_first(df_with)

# Example Check
df1.columns.values.tolist()

['Study_Status_Bin',
 'Funder_Industry_Bin',
 'Funder_Counts_Log',
 'Placebo_Bin',
 'Standard_Care_Bin',
 'Healthy_Bin',
 'Adverse_Counts_Log',
 'Allocation_Bin',
 'Arm_Counts_Log',
 'Enrollment_Counts_Log',
 'City_Counts_Log',
 'Global_Bin',
 'Sex_List_MALE',
 'Age_List_ADULT',
 'Age_List_OLDER_ADULT',
 'Intervention_Type_List_BIOLOGICAL',
 'Intervention_Type_List_DEVICE',
 'Intervention_Type_List_DRUG',
 'Intervention_Type_List_INTERV_UNSPES',
 'Intervention_Type_List_PROCEDURE',
 'Intervention_Route_List_Injection',
 'Intervention_Route_List_Oral',
 'Intervention_Route_List_Surgical',
 'Intervention_Route_List_Topical',
 'Conditions_Detail_List_Bacterial, Mycoses, Virus',
 'Conditions_Detail_List_Cardiovascular',
 'Conditions_Detail_List_Digestive System, Nutritional, Metabolic',
 'Conditions_Detail_List_Endocrine System',
 'Conditions_Detail_List_Eye',
 'Conditions_Detail_List_Hemic, Lymphatic, Immune System',
 'Conditions_Detail_List_Hereditary, Neonatal, Abnormalities',
 'Conditi

# Binary

In [311]:
# Binary Encoding
def fun_bin_enc(dfi, cols):
    dfi = dfi.copy()
    for col in cols:
        cats = CategoricalDtype(categories = sorted(dfi[col].dropna().unique()), ordered = False)
        dfi[col] = dfi[col].astype(cats).cat.codes
    return dfi

### Cols
def fun_bin_cols(dfis):
    bin_cols = []
    for dfi in dfis:
        bin_cols = bin_cols + [[col for col in dfi.columns if '_Categ' in col or '_Bin' in col]] 
    return bin_cols

# Apply
bin_cols = fun_bin_cols(dfis)

df1 = fun_bin_enc(df1, bin_cols[0])
df2 = fun_bin_enc(df2, bin_cols[1])
df3 = fun_bin_enc(df3, bin_cols[2])
df4 = fun_bin_enc(df4, bin_cols[3])
df_with = fun_bin_enc(df_with, bin_cols[4])

# Example
display(df1['Study_Status_Bin'].value_counts())  # Completed = 0, Terminated = 1
display(bin_cols[0])  # bin_cols[0] --> bin_cols of df1
display(df1[bin_cols[0]]) 

Study_Status_Bin
0    18646
1     2366
Name: count, dtype: int64

['Study_Status_Bin',
 'Funder_Industry_Bin',
 'Placebo_Bin',
 'Standard_Care_Bin',
 'Healthy_Bin',
 'Allocation_Bin',
 'Global_Bin']

,Study_Status_Bin,Funder_Industry_Bin,Placebo_Bin,Standard_Care_Bin,Healthy_Bin,Allocation_Bin,Global_Bin
0,0,1,0,0,1,1,0
1,0,0,1,0,1,1,0
5,0,1,0,0,0,0,0
6,0,1,0,0,1,1,0
7,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...
66481,0,1,0,0,0,0,0
66489,0,1,0,0,1,0,0
66492,0,1,0,0,1,1,0
66495,0,0,1,0,0,1,0


# Interactions

In [312]:
dfis = [df1, df2, df3, df4, df_with]
iss = [1, 2, 3, 4, 5]


In [313]:
# Intreaction of Categorical_x_Binary
def fun_inter(dfi, col1, col2, stip):

    cols1 = [col for col in dfi.columns if col1 in col]  # All dfs have the same columns
    cols2 = [col for col in dfi.columns if col2 in col]

    for col1, col2 in product(cols1, cols2):
        inter_col1 = f"{col1}_x_{col2}"
        dfi[inter_col1] = dfi[col1] * dfi[col2]
        inter_col = re.sub(stip, "", inter_col1)
        dfi.rename(columns = {inter_col1: inter_col}, inplace=True)
        
    return dfi
    
df1 = fun_inter(df1, 'Intervention_Type_List', 'Funder_Industry_Bin', r"Intervention_Type_List_|_Bin" )
df2 = fun_inter(df2, 'Intervention_Type_List', 'Funder_Industry_Bin', r"Intervention_Type_List_|_Bin")
df3 = fun_inter(df3, 'Intervention_Type_List', 'Funder_Industry_Bin', r"Intervention_Type_List_|_Bin")
df4 = fun_inter(df4, 'Intervention_Type_List', 'Funder_Industry_Bin', r"Intervention_Type_List_|_Bin")
df_with = fun_inter(df_with, 'Intervention_Type_List', 'Funder_Industry_Bin', r"Intervention_Type_List_|_Bin")

df1 = df1.rename(columns = {col: col + "_Bin" if "_x_Funder_Industry" in col else col for col in df1.columns})
df2 = df2.rename(columns = {col: col + "_Bin" if "_x_Funder_Industry" in col else col for col in df2.columns})
df3 = df3.rename(columns = {col: col + "_Bin" if "_x_Funder_Industry" in col else col for col in df3.columns})
df4 = df4.rename(columns = {col: col + "_Bin" if "_x_Funder_Industry" in col else col for col in df4.columns})
df_with = df_with.rename(columns = {col: col + "_Bin" if "_x_Funder_Industry" in col else col for col in df_with.columns})

df1 = fun_inter(df1, 'Enrollment_Counts_Log', 'Conditions_Detail_List', r"_Conditions_Detail_List|_Counts")
df2 = fun_inter(df2, 'Enrollment_Counts_Log', 'Conditions_Detail_List', r"_Conditions_Detail_List|_Counts")
df3 = fun_inter(df3, 'Enrollment_Counts_Log', 'Conditions_Detail_List', r"_Conditions_Detail_List|_Counts")
df4 = fun_inter(df4, 'Enrollment_Counts_Log', 'Conditions_Detail_List', r"_Conditions_Detail_List|_Counts")
df_with = fun_inter(df_with, 'Enrollment_Counts_Log', 'Conditions_Detail_List', r"_Conditions_Detail_List|_Counts")

df1 = fun_inter(df1, 'Enrollment_Counts_Log', 'City_Counts_Log', r"_Counts")
df2 = fun_inter(df2, 'Enrollment_Counts_Log', 'City_Counts_Log', r"_Counts")
df3 = fun_inter(df3, 'Enrollment_Counts_Log', 'City_Counts_Log', r"_Counts")
df4 = fun_inter(df4, 'Enrollment_Counts_Log', 'City_Counts_Log', r"_Counts")
df_with = fun_inter(df_with, 'Enrollment_Counts_Log', 'City_Counts_Log', r"_Counts")



In [314]:
dfis = [df1, df2, df3, df4, df_with]
iss = [1, 2, 3, 4, 5]

# Pivot
pivots_inter = []

for i, dfi in zip(iss, dfis):
    pivot = dfi.pivot_table(
        index = "Study_Status_Bin",
        values = [col for col in dfi.columns if '_x_' in col],
        aggfunc = "sum", 
        observed = False)
    
    pivot_inter_1 = pivot.T
    pivot_inter_1.columns = [f"df{i}_{outcome}" for outcome in pivot_inter_1.columns]  # Optional: label by df index
    
    pivots_inter.append(pivot_inter_1)

pivot_inter1 = pivots_inter[0]
pivot_inter2 = pivots_inter[1]
pivot_inter3 = pivots_inter[2]
pivot_inter4 = pivots_inter[3]
pivot_inter5 = pivots_inter[4]

# Checks
inter_cols = [col for col in df4.columns if '_x_' in col]  # i have the world other in too many data levels !!
display(len(inter_cols))  # Must have created 1

pivot_inter = pd.concat(pivots_inter, axis=1)
pivot_inter['Sum_Counts'] = pivot_inter.sum(axis = 1)
pivot_inter.sort_values(by = 'Sum_Counts')


23

,df1_0,df1_1,df2_0,df2_1,df3_0,df3_1,df4_0,df4_1,df5_0,df5_1,Sum_Counts
PROCEDURE_x_Funder_Industry_Bin,43.000000,8.000000,61.000000,12.000000,78.000000,9.000000,26.000000,6.000000,208.000000,17.000000,468.000000
INTERV_UNSPES_x_Funder_Industry_Bin,185.000000,30.000000,189.000000,50.000000,143.000000,28.000000,67.000000,9.000000,584.000000,37.000000,1322.000000
DEVICE_x_Funder_Industry_Bin,148.000000,13.000000,112.000000,26.000000,181.000000,34.000000,185.000000,33.000000,626.000000,42.000000,1400.000000
BIOLOGICAL_x_Funder_Industry_Bin,1103.000000,183.000000,872.000000,170.000000,1125.000000,88.000000,264.000000,10.000000,3364.000000,147.000000,7326.000000
"Enrollment_Log_x_Stomatognathic, Otorhinolaryngologic",745.153986,54.493132,1900.551788,193.222819,1845.361899,154.955401,1653.536692,78.864527,6519.401587,0.000000,13145.541832
Enrollment_Log_x_Eye,749.630629,67.347213,1955.645271,278.568113,2286.760985,232.137103,1606.864031,115.778821,6878.084407,0.000000,14170.816572
"Enrollment_Log_x_Hereditary, Neonatal, Abnormalities",2261.838838,228.068314,3537.257122,711.843246,3085.687150,521.624964,1160.163549,129.047277,10481.148686,0.000000,22116.679146
"Enrollment_Log_x_Musculoskeletal, Neural",1584.368321,124.313667,2950.884849,493.442633,3387.299881,387.481978,2412.832795,253.805262,11580.053434,3.433987,23177.916808
Enrollment_Log_x_Endocrine System,4103.751918,504.810679,3970.464106,681.457555,4474.432741,545.894536,3571.477650,272.930178,18591.806427,9.040145,36726.065936
Enrollment_Log_x_Cardiovascular,2979.120244,404.301888,4842.209648,807.188161,5158.441190,873.246800,5659.260545,661.858105,20905.121333,14.072809,42304.820722


In [315]:
def fun_zeros(pivot_merged, count, missing):

    num_cols = pivot_merged.select_dtypes(include='number')
    mask = num_cols < count
    if missing == True:
        mask |= num_cols.isna()
    sparse = pivot_merged[mask.any(axis=1)]
    return sparse

fun_zeros(pivot_inter, 20, False)

,df1_0,df1_1,df2_0,df2_1,df3_0,df3_1,df4_0,df4_1,df5_0,df5_1,Sum_Counts
BIOLOGICAL_x_Funder_Industry_Bin,1103.000000,183.000000,872.000000,170.000000,1125.000000,88.000000,264.000000,10.000000,3364.000000,147.000000,7326.000000
DEVICE_x_Funder_Industry_Bin,148.000000,13.000000,112.000000,26.000000,181.000000,34.000000,185.000000,33.000000,626.000000,42.000000,1400.000000
"Enrollment_Log_x_Bacterial, Mycoses, Virus",6269.984841,474.981002,7752.434272,1239.145498,10121.607545,868.418549,5883.286319,553.978545,36322.457523,0.000000,69486.294094
Enrollment_Log_x_Cardiovascular,2979.120244,404.301888,4842.209648,807.188161,5158.441190,873.246800,5659.260545,661.858105,20905.121333,14.072809,42304.820722
"Enrollment_Log_x_Digestive System, Nutritional, Metabolic",9321.898883,1054.485996,11448.299388,1999.104048,10969.027270,1437.341823,7752.876468,750.685616,45943.168431,9.040145,90685.928068
Enrollment_Log_x_Endocrine System,4103.751918,504.810679,3970.464106,681.457555,4474.432741,545.894536,3571.477650,272.930178,18591.806427,9.040145,36726.065936
Enrollment_Log_x_Eye,749.630629,67.347213,1955.645271,278.568113,2286.760985,232.137103,1606.864031,115.778821,6878.084407,0.000000,14170.816572
"Enrollment_Log_x_Hemic, Lymphatic, Immune System",7610.379045,1423.997375,9707.220554,2021.755568,8621.325389,1103.792642,3897.506327,438.862167,31914.765548,3.555348,66743.159963
"Enrollment_Log_x_Hereditary, Neonatal, Abnormalities",2261.838838,228.068314,3537.257122,711.843246,3085.687150,521.624964,1160.163549,129.047277,10481.148686,0.000000,22116.679146
"Enrollment_Log_x_Musculoskeletal, Neural",1584.368321,124.313667,2950.884849,493.442633,3387.299881,387.481978,2412.832795,253.805262,11580.053434,3.433987,23177.916808


In [316]:
def fun_drop_inter(dfi):
    cols_to_merge = [#'BEHAVIORAL_x_Funder_Industry_Bin', # 'DIETARY_SUPPLEMENT_x_Funder_Industry_Bin', 
        'PROCEDURE_x_Funder_Industry_Bin', 
        'INTERV_UNSPES_x_Funder_Industry_Bin', 'BIOLOGICAL_x_Funder_Industry_Bin',
        'DEVICE_x_Funder_Industry_Bin']

    dfi = dfi.drop(columns = cols_to_merge, axis=1)
    return dfi

df1 = fun_drop_inter(df1)
df2 = fun_drop_inter(df2)
df3 = fun_drop_inter(df3)
df4 = fun_drop_inter(df4)
df_with = fun_drop_inter(df_with)

df_with[[col for col in df_with.columns if '_x_Funder' in col]].columns.to_list()

['DRUG_x_Funder_Industry_Bin']

In [317]:
def fun_drop_cont(dfi):
    cols_cont= [col for col in dfi.columns if 'Continent' in col]
    dfi = dfi.drop(columns = cols_cont)
    return dfi

df1 = fun_drop_cont(df1)
df2 = fun_drop_cont(df2)
df3 = fun_drop_cont(df3)
df4 = fun_drop_cont(df4)
df_with = fun_drop_cont(df_with)

# Save Dfs

In [318]:
df1.to_pickle(r".\df_dummies\df1_dummies.pkl")
df2.to_pickle(r".\df_dummies\df2_dummies.pkl")
df3.to_pickle(r".\df_dummies\df3_dummies.pkl")
df4.to_pickle(r".\df_dummies\df4_dummies.pkl")
df_with.to_pickle(r".\df_dummies\df_with_dummies.pkl")


In [319]:
display(df1.shape)
display(df2.shape)
display(df3.shape)
display(df4.shape)
display(df_with.shape)


(21012, 73)

(19747, 73)

(13317, 73)

(12420, 73)

(60717, 73)

In [320]:
df_with.columns.difference(df1.columns)

Index([], dtype='object')